In [1]:
#Cargando el dataset de las canciones de spotify:

import pandas as pd 

df = pd.read_csv("dataset.csv")
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [2]:
#Translating dataset (songs)

df = df.dropna(subset=["track_name", "artists"])

text = (df["track_name"] + " by " + df["artists"])
text = " ".join(text.tolist())

print(text[:500])

Comedy by Gen Hoshino Ghost - Acoustic by Ben Woodward To Begin Again by Ingrid Michaelson;ZAYN Can't Help Falling In Love by Kina Grannis Hold On by Chord Overstreet Days I Will Remember by Tyrone Wells Say Something by A Great Big World;Christina Aguilera I'm Yours by Jason Mraz Lucky by Jason Mraz;Colbie Caillat Hunger by Ross Copperman Give Me Your Forever by Zack Tabudlo I Won't Give Up by Jason Mraz Solo by Dan Berk Bad Liar by Anna Hamilton Hold On - Remix by Chord Overstreet;Deepend Fall


In [3]:
#Crear el tokenizer:

from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")


In [5]:
#Creando la clase del dataset dataset

from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_lenght, stride):
        self.input_ids = []
        self.target_ids = []
        
        tokens_ids=tokenizer.encode(text)
        for i in range(0, len(tokens_ids) - max_lenght, stride):
            input_chunk = tokens_ids[i:i+max_lenght]
            target_chunks = tokens_ids[i +1: i + max_lenght +1]
            
            self.input_ids.append(input_chunk)
            self.target_ids.append(tokens_ids)
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
    

In [6]:
#Creamos directament el dataset:

dataset = GPTDatasetV1(text, tokenizer, max_lenght=2, stride=2)

Token indices sequence length is longer than the specified maximum sequence length for this model (1409696 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
#Creando el dataloader:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=2, shuffle=False)